# 02 - Bronze Auto Loader Ingestion
## 02 - Weather Ingestion
### Goal
- copy raw files into a governed landing volume
- ingest files into a bronze Delta tables using Auto Loader

In [0]:
%run ../01_setup/00_config

In [0]:
dbutils.fs.mkdirs(weather_path)

for file_info in dbutils.fs.ls(f"{repo_sample_path}/weather"):
    if file_info.name.startswith("weather") and file_info.name.endswith(".csv"):
        try:
            dbutils.fs.cp(file_info.path, f"{weather_path}/{file_info.name}")
            print(f"Copied: {file_info.name}")
        except Exception as e:
            print(f"Copy skipped or failed: {file_info.name} - {e}")

display(dbutils.fs.ls(weather_path))


In [0]:
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", checkpoint_weather_path)
    .load(weather_path)
)

In [0]:
query = (
    bronze_stream_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_weather_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_weather_table)
)

query.awaitTermination()
print("Bronze ingestion complete.")
display(spark.table(bronze_weather_table))